# Real Case Study Analysis

This notebook contains the computational workflow used for the methodological real case study associated with:

**A Framework for Reliability Assessment of Binary Classification Models in Small Tabular Datasets**

The participant level dataset is not included in the public repository. To reproduce this analysis, an authorized copy of the dataset must be placed in the same folder as this notebook using the filename specified below.

The analysis uses sex, nutritional status, age group, and Gijón social scale as predictors. The operational outcome is derived from the `EWGSOP2` variable. Variables used to define the outcome are not included as predictors.

## 1. Setup

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from firthmodels import FirthLogisticRegression

DATA_PATH = Path("SARCOPENIA_ADULTOMAYOR_DATOS.csv")
B = 1000
SEED = 42

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "The participant level dataset is not included in the public repository. "
        "Place an authorized copy named 'SARCOPENIA_ADULTOMAYOR_DATOS.csv' "
        "in the same folder as this notebook to reproduce the real case analysis."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)

## 2. Data preparation

In [ ]:
required_columns = ["SEXO", "MNA", "EDAD", "ESCGIJON", "EWGSOP2"]

missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df_clean = df.dropna(how="all").copy()

df_model = df_clean[
    ["SEXO", "MNA", "EDAD", "ESCGIJON", "EWGSOP2"]
].copy()

df_model["Y"] = df_model["EWGSOP2"].map({
    "Sin riesgo a sarcopenia": 0,
    "Sarcopenia confirmada- severa": 1,
})

df_model["MNA_BIN"] = df_model["MNA"].map({
    "Estado nutricional normal": "Normal",
    "Riesgo a malnutrición": "Riesgo",
    "Malnutricion": "Riesgo",
})

df_model["EDAD_BIN"] = df_model["EDAD"].replace({
    "60-70 años": "60-70",
    "71-80 años": "71-80",
    "81-90 años": "81 o más",
    "más de 90 años": "81 o más",
})

df_model = df_model[
    ["SEXO", "MNA_BIN", "EDAD_BIN", "ESCGIJON", "Y"]
].dropna().copy()

X = df_model[
    ["SEXO", "MNA_BIN", "EDAD_BIN", "ESCGIJON"]
].copy()

y = df_model["Y"].astype(int).copy()

X_encoded = pd.get_dummies(
    X,
    drop_first=True,
    dtype=int,
)

print("Analysis sample:", len(df_model))
print("Encoded predictors:", X_encoded.columns.tolist())
print("Outcome distribution:")
print(y.value_counts())

## 3. Conventional Logit diagnostic

The conventional logistic model is fitted as a diagnostic step. In the manuscript, this model was not retained for the final real case reliability comparison because the small dataset produced unstable estimation associated with separation.

In [ ]:
X_logit = sm.add_constant(X_encoded, has_constant="add")
logit_model = sm.Logit(y, X_logit)

with warnings.catch_warnings(record=True) as captured_warnings:
    warnings.simplefilter("always")

    try:
        logit_result = logit_model.fit(
            disp=False,
            maxiter=200,
        )
        print("Converged:", logit_result.mle_retvals["converged"])
        print(logit_result.summary())
    except Exception as error:
        print("Conventional Logit could not be fitted reliably.")
        print("Error type:", type(error).__name__)
        print("Message:", error)

    if captured_warnings:
        print("\nWarnings:")
        for warning in captured_warnings:
            print(type(warning.message).__name__, ":", warning.message)

## 4. Bootstrap reliability assessment

In [ ]:
rng = np.random.default_rng(SEED)
n = len(X_encoded)

results = []
predictions = {
    "Firth": [],
    "Elastic Net": [],
    "CART": [],
}
firth_convergence = []

for b in range(B):
    idx_boot = rng.choice(
        np.arange(n),
        size=n,
        replace=True,
    )

    idx_oob = np.setdiff1d(
        np.arange(n),
        np.unique(idx_boot),
    )

    X_boot = X_encoded.iloc[idx_boot]
    y_boot = y.iloc[idx_boot]
    X_oob = X_encoded.iloc[idx_oob]
    y_oob = y.iloc[idx_oob]

    if (
        y_boot.nunique() < 2
        or len(idx_oob) == 0
        or y_oob.nunique() < 2
    ):
        continue

    # Firth
    try:
        firth = FirthLogisticRegression(max_iter=500)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            firth.fit(X_boot, y_boot)

        firth_convergence.append(bool(firth.converged_))

        if firth.converged_:
            prob_oob = firth.predict_proba(X_oob)[:, 1]
            prob_total = firth.predict_proba(X_encoded)[:, 1]

            results.append({
                "repetition": b,
                "model": "Firth",
                "AUC": roc_auc_score(y_oob, prob_oob),
                "Brier": brier_score_loss(y_oob, prob_oob),
            })
            predictions["Firth"].append(prob_total)

    except Exception:
        firth_convergence.append(False)

    # Elastic Net
    try:
        elastic = LogisticRegression(
            solver="saga",
            l1_ratio=0.5,
            C=1.0,
            max_iter=5000,
            random_state=SEED,
        )
        elastic.fit(X_boot, y_boot)

        prob_oob = elastic.predict_proba(X_oob)[:, 1]
        prob_total = elastic.predict_proba(X_encoded)[:, 1]

        results.append({
            "repetition": b,
            "model": "Elastic Net",
            "AUC": roc_auc_score(y_oob, prob_oob),
            "Brier": brier_score_loss(y_oob, prob_oob),
        })
        predictions["Elastic Net"].append(prob_total)

    except Exception:
        pass

    # CART
    try:
        cart = DecisionTreeClassifier(
            max_depth=3,
            min_samples_leaf=5,
            random_state=SEED,
        )
        cart.fit(X_boot, y_boot)

        prob_oob = cart.predict_proba(X_oob)[:, 1]
        prob_total = cart.predict_proba(X_encoded)[:, 1]

        results.append({
            "repetition": b,
            "model": "CART",
            "AUC": roc_auc_score(y_oob, prob_oob),
            "Brier": brier_score_loss(y_oob, prob_oob),
        })
        predictions["CART"].append(prob_total)

    except Exception:
        pass

bootstrap_results = pd.DataFrame(results)

print("Valid evaluations:")
print(bootstrap_results["model"].value_counts())

if firth_convergence:
    print(
        "Firth convergence (%):",
        round(np.mean(firth_convergence) * 100, 2),
    )

## 5. Performance and prediction stability

In [ ]:
performance = (
    bootstrap_results
    .groupby("model")
    .agg(
        Valid_repetitions=("AUC", "count"),
        AUC_mean=("AUC", "mean"),
        AUC_SD=("AUC", "std"),
        Brier_mean=("Brier", "mean"),
        Brier_SD=("Brier", "std"),
    )
    .reset_index()
)

prediction_stability_rows = []

for model, model_predictions in predictions.items():
    matrix = np.asarray(model_predictions)

    if matrix.size == 0:
        prediction_sd = np.nan
    else:
        observation_sd = np.std(matrix, axis=0, ddof=1)
        prediction_sd = np.mean(observation_sd)

    prediction_stability_rows.append({
        "model": model,
        "Prediction_SD": prediction_sd,
    })

prediction_stability = pd.DataFrame(prediction_stability_rows)

profile = performance.merge(
    prediction_stability,
    on="model",
    how="left",
)

print(profile.round(4))

## 6. Pareto dominance

In [ ]:
def dominates(a, b):
    conditions = [
        a["AUC_mean"] >= b["AUC_mean"],
        a["AUC_SD"] <= b["AUC_SD"],
        a["Brier_mean"] <= b["Brier_mean"],
        a["Brier_SD"] <= b["Brier_SD"],
        a["Prediction_SD"] <= b["Prediction_SD"],
    ]

    improvements = [
        a["AUC_mean"] > b["AUC_mean"],
        a["AUC_SD"] < b["AUC_SD"],
        a["Brier_mean"] < b["Brier_mean"],
        a["Brier_SD"] < b["Brier_SD"],
        a["Prediction_SD"] < b["Prediction_SD"],
    ]

    return all(conditions) and any(improvements)


dominated_models = set()

for i in profile.index:
    for j in profile.index:
        if i == j:
            continue

        if dominates(profile.loc[i], profile.loc[j]):
            dominated_models.add(profile.loc[j, "model"])

profile["Status"] = profile["model"].apply(
    lambda model: "Dominated"
    if model in dominated_models
    else "Non dominated"
)

print(profile.round(4).to_string(index=False))

## 7. Export

The output contains aggregate model level results only. It does not contain participant level records.

In [ ]:
profile.to_csv(
    "real_case_reliability_profile.csv",
    index=False,
)

print("Saved: real_case_reliability_profile.csv")